In [1]:
from pathlib import Path

import polars as pl
from mcap.reader import make_reader

ROOT = Path("../data/raw/robotis_sample")
mcap_path = next(ROOT.rglob("*.mcap"))
print(mcap_path)

../data/raw/robotis_sample/1_0.mcap


In [2]:
with open(mcap_path, "rb") as f:
    summary = make_reader(f).get_summary()

stats = summary.statistics
print("total messages:", stats.message_count)
print("start (ns):", stats.message_start_time)
print("end   (ns):", stats.message_end_time)
print("duration (s):", (stats.message_end_time - stats.message_start_time) / 1e9)
print()

for channel_id, channel in summary.channels.items():
    schema = summary.schemas[channel.schema_id]
    count = stats.channel_message_counts.get(channel_id, 0)
    print(f"{channel.topic:65s} {schema.name:35s} {count}")

total messages: 10714
start (ns): 1768808157814258629
end   (ns): 1768808166281204539
duration (s): 8.46694591

/zed/zed_node/left/image_rect_color/compressed                    sensor_msgs/msg/CompressedImage     129
/camera_left/camera_left/color/image_rect_raw/compressed          sensor_msgs/msg/CompressedImage     257
/camera_right/camera_right/color/image_rect_raw/compressed        sensor_msgs/msg/CompressedImage     257
/joint_states                                                     sensor_msgs/msg/JointState          1713
/leader/joint_trajectory_command_broadcaster_left/joint_trajectory trajectory_msgs/msg/JointTrajectory 1712
/leader/joint_trajectory_command_broadcaster_right/joint_trajectory trajectory_msgs/msg/JointTrajectory 1713
/leader/joystick_controller_left/joint_trajectory                 trajectory_msgs/msg/JointTrajectory 856
/leader/joystick_controller_right/joint_trajectory                trajectory_msgs/msg/JointTrajectory 1712
/tf                              

In [3]:
rows = []
with open(mcap_path, "rb") as f:
    for schema, channel, message in make_reader(f).iter_messages():
        rows.append({
            "topic": channel.topic,
            "log_time": message.log_time,
            "size_bytes": len(message.data),
        })

msgs = pl.DataFrame(rows)
print(msgs.shape)
msgs.head()

(10714, 3)


topic,log_time,size_bytes
str,i64,i64
"""/camera_right/camera_right/col…",1768808157814258629,392
"""/zed/zed_node/rgb/camera_info""",1768808157814380200,384
"""/leader/joystick_controller_le…",1768808157814396072,104
"""/zed/zed_node/left/image_rect_…",1768808157814752207,288603
"""/zed/zed_node/right/camera_inf…",1768808157814934899,384


In [4]:
rates = (
    msgs.sort(["topic", "log_time"])
    .with_columns(pl.col("log_time").diff().over("topic").alias("dt_ns"))
    .group_by("topic")
    .agg(
        pl.len().alias("messages"),
        (pl.col("dt_ns").median() / 1e6).alias("median_gap_ms"),
        (pl.col("dt_ns").max() / 1e6).alias("max_gap_ms"),
        (pl.col("dt_ns").std() / 1e6).alias("jitter_ms"),
    )
    .with_columns((1000 / pl.col("median_gap_ms")).round(1).alias("approx_hz"))
    .sort("messages", descending=True)
)
rates

topic,messages,median_gap_ms,max_gap_ms,jitter_ms,approx_hz
str,u32,f64,f64,f64,f64
"""/tf""",1714,0.272613,15.4255,5.134201,3668.2
"""/leader/joint_trajectory_comma…",1713,0.6027485,17.634025,5.227035,1659.1
"""/joint_states""",1713,1.04983,15.820548,5.100071,952.5
"""/leader/joint_trajectory_comma…",1712,1.219769,24.954911,5.216695,819.8
"""/leader/joystick_controller_ri…",1712,0.979796,17.952976,5.144879,1020.6
…,…,…,…,…,…
"""/zed/zed_node/left/image_rect_…",129,66.673012,101.334682,9.64142,15.0
"""/zed/zed_node/left/camera_info""",129,66.646292,101.247289,9.600788,15.0
"""/camera_left/camera_left/color…",129,67.198319,104.869219,11.884923,14.9


In [5]:
from mcap_ros2.decoder import DecoderFactory

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/joint_states"]):
        print("log_time (ns):", message.log_time)
        print("header stamp:", ros_msg.header.stamp.sec, "s +", ros_msg.header.stamp.nanosec, "ns")
        print("joint names:", list(ros_msg.name))
        print("positions:", list(ros_msg.position))
        break

log_time (ns): 1768808157815052470
header stamp: 1768808157 s + 716013770 ns
joint names: ['arm_l_joint1', 'arm_l_joint2', 'arm_l_joint3', 'arm_l_joint4', 'arm_l_joint5', 'arm_l_joint6', 'arm_l_joint7', 'arm_r_joint1', 'arm_r_joint2', 'arm_r_joint3', 'arm_r_joint4', 'arm_r_joint5', 'arm_r_joint6', 'arm_r_joint7', 'gripper_l_joint1', 'gripper_r_joint1', 'head_joint1', 'head_joint2', 'left_wheel_drive', 'left_wheel_steer', 'lift_joint', 'rear_wheel_drive', 'rear_wheel_steer', 'right_wheel_drive', 'right_wheel_steer']
positions: [0.09659285273717397, 0.00044341632149819316, 0.5874906733103898, -0.33132786595839314, -0.47093210188089185, -0.26142388208544826, -0.04451494578123612, 0.12576245615681186, -0.04140549704800695, -0.06442719309119693, -0.28532042654672923, 0.1610440142781816, -0.06508632546099154, -0.05213750968476098, 0.13652429012182207, 0.12885438618239387, 0.8191457407307254, 0.34361169648617684, -0.0020972393584374, -3.5952674716069715e-05, -0.012190000000000001, 0.001498028

In [6]:
gaps = []
with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/joint_states"]):
        stamp_ns = ros_msg.header.stamp.sec * 1_000_000_000 + ros_msg.header.stamp.nanosec
        gaps.append(message.log_time - stamp_ns)

delay_ms = pl.Series("log_minus_stamp_ms", gaps) / 1e6
delay_ms.describe()

statistic,value
str,f64
"""count""",1713.0
"""null_count""",0.0
"""mean""",3.037647
"""std""",6.411004
"""min""",0.206724
"""25%""",1.251738
"""50%""",1.866023
"""75%""",4.159637
"""max""",99.0387


In [7]:
duration_s = (stats.message_end_time - stats.message_start_time) / 1e9

rates.with_columns(
    (pl.col("messages") / duration_s).round(1).alias("true_avg_hz")
).select("topic", "messages", "true_avg_hz", "approx_hz", "max_gap_ms")

topic,messages,true_avg_hz,approx_hz,max_gap_ms
str,u32,f64,f64,f64
"""/tf""",1714,202.4,3668.2,15.4255
"""/leader/joint_trajectory_comma…",1713,202.3,1659.1,17.634025
"""/joint_states""",1713,202.3,952.5,15.820548
"""/leader/joint_trajectory_comma…",1712,202.2,819.8,24.954911
"""/leader/joystick_controller_ri…",1712,202.2,1020.6,17.952976
…,…,…,…,…
"""/zed/zed_node/left/image_rect_…",129,15.2,15.0,101.334682
"""/zed/zed_node/left/camera_info""",129,15.2,15.0,101.247289
"""/camera_left/camera_left/color…",129,15.2,14.9,104.869219


In [8]:
stamps = []
with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/joint_states"]):
        stamps.append(ros_msg.header.stamp.sec * 1_000_000_000 + ros_msg.header.stamp.nanosec)

stamp_gaps_ms = pl.Series("stamp_gap_ms", stamps).diff() / 1e6
print("negative gaps (out-of-order stamps):", (stamp_gaps_ms < 0).sum())
stamp_gaps_ms.describe()

negative gaps (out-of-order stamps): 0


statistic,value
str,f64
"""count""",1712.0
"""null_count""",1.0
"""mean""",5.000002
"""std""",5.001664
"""min""",0.0
"""25%""",0.0
"""50%""",9.78164
"""75%""",9.997357
"""max""",10.240625


In [9]:
from collections import Counter

name_groups = Counter()
msgs_per_stamp = Counter()

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/joint_states"]):
        name_groups[tuple(ros_msg.name)] += 1
        stamp_ns = ros_msg.header.stamp.sec * 1_000_000_000 + ros_msg.header.stamp.nanosec
        msgs_per_stamp[stamp_ns] += 1

print("distinct joint groups:", len(name_groups))
for names, count in name_groups.items():
    print(f"  {count} messages, {len(names)} joints: {names[0]} … {names[-1]}")

print("messages per timestamp:", Counter(msgs_per_stamp.values()))

distinct joint groups: 1
  1713 messages, 25 joints: arm_l_joint1 … right_wheel_steer
messages per timestamp: Counter({2: 856, 1: 1})


In [10]:
first_seen = {}
identical = 0
conflicting = 0

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/joint_states"]):
        stamp_ns = ros_msg.header.stamp.sec * 1_000_000_000 + ros_msg.header.stamp.nanosec
        positions = tuple(ros_msg.position)
        if stamp_ns in first_seen:
            if first_seen[stamp_ns] == positions:
                identical += 1
            else:
                conflicting += 1
        else:
            first_seen[stamp_ns] = positions

print("identical duplicates:", identical)
print("same timestamp, different values:", conflicting)

identical duplicates: 856
same timestamp, different values: 0


In [11]:
rates.filter(pl.col("topic").str.contains("compressed")).with_columns(
    (pl.col("messages") / duration_s).round(1).alias("true_avg_hz")
).select("topic", "messages", "true_avg_hz")

topic,messages,true_avg_hz
str,u32,f64
"""/camera_right/camera_right/col…",257,30.4
"""/camera_left/camera_left/color…",257,30.4
"""/zed/zed_node/left/image_rect_…",129,15.2
